In [1]:
import pandas as pd
import os

KG_DIR      = r'D:\Memoire\LLMKGER\kg_final'
RECBOLE_DIR = r'D:\Memoire\LLMKGER\recbole_data\llmkger'
os.makedirs(RECBOLE_DIR, exist_ok=True)

# Charger toutes les interactions
train = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
valid = pd.read_csv(os.path.join(KG_DIR, 'interactions_valid.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
test  = pd.read_csv(os.path.join(KG_DIR, 'interactions_test.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])

all_inter = pd.concat([train, valid, test], ignore_index=True)
print(f'Interactions totales : {len(all_inter):,}')
print(f'Users : {all_inter["user_id"].nunique():,}')
print(f'Items : {all_inter["item_id"].nunique():,}')


Interactions totales : 5,258,360
Users : 106,735
Items : 2,822


In [2]:
# ── Filtrer : garder users avec 10-80 interactions ────────────────────────
MIN_INTER = 10
MAX_INTER = 80   # plafonner pour éviter les power-users
MAX_USERS = 15000  # limiter le nombre total de users

user_counts = all_inter.groupby('user_id').size()
valid_users = user_counts[(user_counts >= MIN_INTER) & (user_counts <= MAX_INTER)].index

print(f'Users avec {MIN_INTER}-{MAX_INTER} interactions : {len(valid_users):,}')

# Échantillonner MAX_USERS users au hasard
import numpy as np
np.random.seed(42)
if len(valid_users) > MAX_USERS:
    sampled_users = np.random.choice(valid_users, MAX_USERS, replace=False)
else:
    sampled_users = valid_users

filtered = all_inter[all_inter['user_id'].isin(sampled_users)].copy()

print(f'\nAprès filtrage :')
print(f'  Interactions : {len(filtered):,}')
print(f'  Users : {filtered["user_id"].nunique():,}')
print(f'  Items : {filtered["item_id"].nunique():,}')
print(f'  Moyenne : {filtered.groupby("user_id").size().mean():.1f} interactions/user')


Users avec 10-80 interactions : 104,233

Après filtrage :
  Interactions : 702,362
  Users : 15,000
  Items : 2,708
  Moyenne : 46.8 interactions/user


In [3]:
# ── Écrire le fichier .inter (version rapide avec to_csv) ──────────────────
out = filtered[['user_id', 'item_id']].copy()
out['rating'] = 1.0
out.columns = ['user_id:token', 'item_id:token', 'rating:float']

inter_path = os.path.join(RECBOLE_DIR, 'llmkger.inter')
out.to_csv(inter_path, sep='\t', index=False)

size = os.path.getsize(inter_path) / 1024 / 1024
print(f'✅ Fichier .inter créé : {len(out):,} interactions, {size:.1f} MB')
print(f'→ {inter_path}')


✅ Fichier .inter créé : 702,362 interactions, 16.8 MB
→ D:\Memoire\LLMKGER\recbole_data\llmkger\llmkger.inter


In [4]:
# ── Recréer aussi le fichier .kg et .link pour KGAT (filtrés) ──────────────
df_kg = pd.read_csv(os.path.join(KG_DIR, 'kg_train.tsv'),
                    sep='\t', header=0, names=['head','relation','tail'])
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()

# Items présents après filtrage
items_kept = set(filtered['item_id'].unique())

# .kg : garder triplets dont head OU tail est lié aux items gardés
# (simplification : on garde tout le KG sémantique)
kg_out = df_kg_sem.copy()
kg_out.columns = ['head_id:token', 'relation_id:token', 'tail_id:token']
kg_path = os.path.join(RECBOLE_DIR, 'llmkger.kg')
kg_out.to_csv(kg_path, sep='\t', index=False)
print(f'✅ .kg créé : {len(kg_out):,} triplets')

# .link : item → entity
link_out = pd.DataFrame({'item_id:token': list(items_kept),
                          'entity_id:token': list(items_kept)})
link_path = os.path.join(RECBOLE_DIR, 'llmkger.link')
link_out.to_csv(link_path, sep='\t', index=False)
print(f'✅ .link créé : {len(link_out):,} liens')


✅ .kg créé : 648,349 triplets
✅ .link créé : 2,708 liens


In [5]:
import os, glob

RECBOLE_DIR = r'D:\Memoire\LLMKGER\recbole_data\llmkger'

# Supprimer les fichiers de cache .pth créés par RecBole
cache_files = glob.glob(os.path.join(RECBOLE_DIR, '*.pth'))
cache_files += glob.glob(os.path.join(RECBOLE_DIR, '*dataset*'))

for f in cache_files:
    if os.path.isfile(f) and f.endswith('.pth'):
        os.remove(f)
        print(f'Supprimé : {os.path.basename(f)}')

# Vérifier le contenu du dossier
print('\nContenu actuel du dossier :')
for f in os.listdir(RECBOLE_DIR):
    size = os.path.getsize(os.path.join(RECBOLE_DIR, f)) / 1024 / 1024
    print(f'  {f} : {size:.1f} MB')


Contenu actuel du dossier :
  llmkger.inter : 16.8 MB
  llmkger.kg : 40.4 MB
  llmkger.link : 0.1 MB
